In [6]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch 

from torch.utils.data import TensorDataset,DataLoader
import torch.nn as nn
import torch.nn.functional as F

In [13]:
from functions import testloader
fn='../data/DRIFT_DATA_TEST.csv'
inputs= ['sin','cos','x_EASE','y_EASE','windnorm','u_ERA5','v_ERA5','sic_CDR', 'h_piomas','bath']
datapd,dataset,dataloader,means,stds,maxes,labels=testloader(filename=fn,inputlist=inputs,target='buoynorm',trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')
print(labels)

Bath size is the full test set
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
[Index(['buoynorm'], dtype='object'), Index(['x_EASE', 'y_EASE', 'u_ERA5', 'v_ERA5', 'sic_CDR', 'h_piomas', 'sin',
       'cos', 'bath', 'windnorm'],
      dtype='object')]


In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F


class staticNN(nn.Module):
    """Hummmm"""
    def __init__(self, n_inputs):
        """MLP with Relu, 256->128->64
        linear 64->2"""
        super().__init__()
        self.fc1=nn.Linear(n_inputs,256)
        self.fc2=nn.Linear(256,128)
        self.fc3=nn.Linear(128,64)
        # self.fc4=nn.Linear(32,16)
        # self.fc5=nn.Linear(16,8)
        # self.fc6=nn.Linear(8,4)
        self.final=nn.Linear(64,1)
        #self.fourier=nn.Linear(n_inputs,n_inputs)

    def forward(self,x):
        #x=torch.cos(self.fourier(x))
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=F.relu(self.fc3(x))
        # x=F.sigmoid(self.fc4(x))
        # x=F.sigmoid(self.fc5(x))
        # x=F.sigmoid(self.fc6(x))
        x=self.final(x)
        return x

staticpd=datapd.drop(['buoynorm'], axis=1)

# Bring it to GPU
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')
staticgpu= staticNN(n_inputs=staticpd.shape[1]).to(device)

staticgpu.load_state_dict(torch.load('../exploration//weights//babygpu_static_10input_999E_weight147.pt', weights_only=True))
staticgpu.eval()  # if you're doing inference, not continuing training

staticity= torch.sigmoid(staticgpu(torch.tensor(staticpd.values, dtype= torch.float32).to(device)).to('cpu').detach()).numpy()

datapd['staticity']=staticity

mask = datapd['staticity'] >= 0.950 # mask for 95% sure of staticity
staticbuoy = datapd[mask].copy() # the static cases
testpd = datapd[~mask].copy() # the moving case
